In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ====================================================================
# 1. LOAD DATA & PREP ML FEATURES
# ====================================================================
df_all = spark.read.table("gold_saturday_predictor_features").toPandas()
df_all['Year'] = df_all['Year'].astype(int)
df_all['Race_Pace_Rank'] = df_all.groupby(['Year', 'Track'])['Pace_Delta_To_Leader'].rank(method='dense')

df_all['Is_Podium_Contender'] = np.where(
    (df_all['Simulated_Grid_Position'] <= 5) & (df_all['Race_Pace_Rank'] <= 5), 1, 0
)

# Time-Series Split
df_history = df_all[df_all['Year'] < 2026]
df_future = df_all[(df_all['Year'] == 2026) & (df_all['Track'] == 'china')].copy() # Target Race

features = ['Simulated_Grid_Position', 'Qualifying_Delta_To_Pole', 'Pace_Delta_To_Leader']
X_history = df_history[features]
y_history = df_history['Is_Podium_Contender']

# ====================================================================
# 2. BRAIN 1: XGBOOST (The Historian)
# ====================================================================
print("🧠 Training XGBoost (Historical Pattern Recognition)...")
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42, eval_metric='logloss')
xgb_model.fit(X_history, y_history)

if not df_future.empty:
    X_live = df_future[features]
    xgb_probs = xgb_model.predict_proba(X_live)[:, 1] 
    df_future['XGBoost_Win_Prob'] = (xgb_probs * 100).round(1)
    print("✅ XGBoost Complete.\n")

# ====================================================================
# 3. BRAIN 2: MONTE CARLO SIMULATOR (The Physicist)
# ====================================================================
def run_monte_carlo(df_race, num_simulations=10000, num_laps=50):
    print(f"🎲 Running {num_simulations} Monte Carlo Race Simulations...")
    base_lap_time = 90.0 # Baseline in seconds (exact number doesn't matter for relative margins)
    win_counts = {driver: 0 for driver in df_race['DriverNumber']}
    simulated_times = {}
    
    for index, row in df_race.iterrows():
        driver = row['DriverNumber']
        # mu = baseline + how much slower they are than the leader
        mu = base_lap_time + row['Pace_Delta_To_Leader'] 
        sigma = row['LapTime_StdDev'] # The Chaos Factor
        grid_pos = row['Simulated_Grid_Position']
        
        # Grid Penalty: Lose 1.5s per position behind pole on Lap 1
        grid_penalty = (grid_pos - 1) * 1.5 
        
        # Simulate 10,000 races of 50 laps instantly
        laps_matrix = np.random.normal(loc=mu, scale=sigma, size=(num_simulations, num_laps))
        total_times = laps_matrix.sum(axis=1) + grid_penalty
        
        simulated_times[driver] = total_times
        
    # Convert to 2D matrix to find the winner of each simulation
    drivers = list(simulated_times.keys())
    times_matrix = np.array([simulated_times[d] for d in drivers])
    
    # Find index of fastest time per simulation
    winning_indices = np.argmin(times_matrix, axis=0)
    
    # Tally wins
    for idx in winning_indices:
        winning_driver = drivers[idx]
        win_counts[winning_driver] += 1
        
    # Calculate Probabilities
    mc_probs = []
    for index, row in df_race.iterrows():
        driver = row['DriverNumber']
        prob = (win_counts[driver] / num_simulations) * 100
        mc_probs.append(prob)
        
    df_race['Monte_Carlo_Win_Prob'] = np.round(mc_probs, 1)
    print("✅ Monte Carlo Complete.\n")
    return df_race

if not df_future.empty:
    df_future = run_monte_carlo(df_future, num_simulations=10000, num_laps=50)

    # ====================================================================
    # 4. THE GAP ANALYSIS (Comparing the Brains)
    # ====================================================================
    print("🏁 2026 GRAND PRIX - DUAL ENGINE PREDICTIONS 🏁")
    print("===========================================================================================")
    
    # Sort by XGBoost to establish our baseline favorite
    predictions = df_future.sort_values(by='XGBoost_Win_Prob', ascending=False)
    
    display(predictions[['Team', 'DriverNumber', 'Simulated_Grid_Position', 'LapTime_StdDev', 'XGBoost_Win_Prob', 'Monte_Carlo_Win_Prob']].head(10))

StatementMeta(, f0ebca63-e86d-403b-9297-4d59b927cebb, 4, Finished, Available, Finished, False)

🧠 Training XGBoost (Historical Pattern Recognition)...


✅ XGBoost Complete.

🎲 Running 10000 Monte Carlo Race Simulations...
✅ Monte Carlo Complete.

🏁 2026 GRAND PRIX - DUAL ENGINE PREDICTIONS 🏁


SynapseWidget(Synapse.DataFrame, c9e764a4-374b-440e-a13e-bc5eb121b5cd)